In [7]:
from backend.enrichment import (
    bootstrap, types, google_places, scoring, geo, Config, web_searches, json_storage,
    places_manager, place_enrichment,
    PLACES_API_KEY, OPENAI_API_KEY, TAVILY_API_KEY
)

In [6]:
geo.meters_to_lng_deg(40.7128, 1000)

0.0021061414923295805

In [2]:
cfg = Config(
    api_key=PLACES_API_KEY,
    lat=40.76894303040112,
    lng=-73.98811816889815,
    radius_m=1200,
    grid_step_m=400,
    nearby_search_radius_m=300,
    include_types=("cafe", "restaurant", "library", "bakery", "lodging"),
    keyword=None,
    max_places_to_enrich=3,      
    request_sleep_s=0.25    
)

In [5]:
d = google_places.nearby_search(cfg, place_type="cafe")

In [4]:
birch_coffee_id = "ChIJhz0V-lhYwokR-0fYpHMII94"

In [4]:
birch_details = google_places.place_details(cfg, birch_coffee_id)

In [5]:
# Test nearby_search with debug
results = google_places.nearby_search(cfg, "cafe", debug=True)

# Test place_details with debug  
details = google_places.place_details(cfg, birch_coffee_id, debug=True)


DEBUG: [nearby_search] Searching for type: cafe
DEBUG: [nearby_search] Location: (40.76894303040112, -73.98811816889815), Radius: 300
DEBUG: [nearby_search] Found 14 results on this page
DEBUG: [nearby_search] First result keys: ['id', 'types', 'formattedAddress', 'location', 'rating', 'websiteUri', 'regularOpeningHours', 'businessStatus', 'priceLevel', 'userRatingCount', 'displayName', 'servesCoffee', 'restroom', 'parkingOptions', 'accessibilityOptions']
DEBUG: [nearby_search] First result id: ChIJ1bM7WeVZwokRdnontikwFK4
DEBUG: [nearby_search] First result displayName: The Oasis Cafe

DEBUG: [nearby_search] Binary attributes in first result:
  - restroom: True (type: bool)
  - servesCoffee: True (type: bool)
  - goodForGroups: None (type: NoneType)
  - parkingOptions: {'paidParkingLot': True, 'paidStreetParking': True} (type: dict)
  - accessibilityOptions: {'wheelchairAccessibleParking': False, 'wheelchairAccessibleEntrance': True} (type: dict)
  - outdoorSeating: None (type: NoneTy

In [10]:
birch_details

{'business_status': 'OPERATIONAL',
 'formatted_address': '884 9th Ave, New York, NY 10019, USA',
 'geometry': {'location': {'lat': 40.7681474, 'lng': -73.98530439999999}},
 'name': 'Birch Coffee',
 'opening_hours': {'open_now': False,
  'periods': [{'close': {'day': 0, 'time': '1600'},
    'open': {'day': 0, 'time': '0730'}},
   {'close': {'day': 1, 'time': '1600'}, 'open': {'day': 1, 'time': '0730'}},
   {'close': {'day': 2, 'time': '1600'}, 'open': {'day': 2, 'time': '0730'}},
   {'close': {'day': 3, 'time': '1600'}, 'open': {'day': 3, 'time': '0730'}},
   {'close': {'day': 4, 'time': '1600'}, 'open': {'day': 4, 'time': '0730'}},
   {'close': {'day': 5, 'time': '1600'}, 'open': {'day': 5, 'time': '0730'}},
   {'close': {'day': 6, 'time': '1600'}, 'open': {'day': 6, 'time': '0730'}}],
  'weekday_text': ['Monday: 7:30\u202fAM\u2009–\u20094:00\u202fPM',
   'Tuesday: 7:30\u202fAM\u2009–\u20094:00\u202fPM',
   'Wednesday: 7:30\u202fAM\u2009–\u20094:00\u202fPM',
   'Thursday: 7:30\u202fAM\

In [ ]:
# e_cfg = web_searches.EnrichmentConfig(
#     max_results_per_query=5,
#     fetch_pages=True,
#     max_pages_to_fetch=4,
#     include_reddit=True,
#     min_unique_sources=3,
#     use_llm=False,  # flip to True and pass llm_client if you wire one
# )

In [5]:
res = web_searches.enrich_place_with_agent(place=birch_details,
place_id=birch_details["place_id"])

print("Attributes:", res.attributes.model_dump(),
"\n")
print("Evidence:")
for ev in res.evidence:
    print(f"- {ev.url}\n  title={ev.title}\n snippet={ev.snippet}\n  quote={ev.quote}\n")
#print("Raw agent JSON (truncated):", res.raw_agent_output[:1200])

Attributes: {'wifi_quality': 'mixed', 'outlet_availability': 'few', 'noise_level': 'mixed', 'laptop_friendly': 'no', 'time_pressure': 'some', 'seating_comfort': 'mixed', 'best_times': ['weekday mornings', 'mid-afternoon (before closing)'], 'common_complaints': ['very small / cramped space', 'limited or no seating', 'can get busy during peak times'], 'notable_positives': ['high-quality coffee', 'friendly, helpful staff', 'convenient grab-and-go for quick visits'], 'wfh_overall': 'no', 'confidence': 0.45, 'summary': 'Great coffee and staff but small, grab-and-go; limited seating/outlets — not ideal for extended laptop work.'} 

Evidence:
- https://m.yelp.com/biz/birch-coffee-new-york-16?start=40
  title=BIRCH COFFEE - Yelp
 snippet=Small shop with just standing room only so mainly for to-go orders. ... It's a take out place with no seating ... Good place to grab and go for a cup of coffee
  quote=Small shop with just standing room only so mainly for to-go orders.

- https://www.tripadvis

In [21]:
# Test the new enrichment system
# Import new modules
from backend.enrichment.json_storage import load_places_json, upsert_place_ids
from backend.enrichment.places_manager import process_nearby_search_sync, process_enrichment_async, get_places_needing_enrichment
from backend.enrichment.google_places import nearby_search_with_sync
from backend.enrichment.place_enrichment import enrich_place_details_sync, enrich_place_web_async

# Set JSON path
JSON_PATH = "backend/data/places_bootstrap.json"

print("New enrichment system imported successfully!")


New enrichment system imported successfully!


In [22]:
from backend.enrichment import reset_enrichment_flag, process_enrichment_async, Config

# 1. Reset ALL places (keep existing enrichment data for reference)
reset_count = reset_enrichment_flag("backend/data/places_bootstrap.json")
print(f"Reset {reset_count} places")

# # 2. Reset specific places
# reset_count = reset_enrichment_flag(
#     "backend/data/places_bootstrap.json",
#     place_ids=["ChIJW6K4fFlYwokRl9znZwmIvEU", "ChIJhz0V-lhYwokR-0fYpHMII94"]
# )

# # 3. Reset and CLEAR enrichment data (start fresh)
# reset_count = reset_enrichment_flag(
#     "backend/data/places_bootstrap.json",
#     clear_enrichment_data=True
# )


DEBUG: File exists, size: 396229 bytes
DEBUG: Read 393581 characters from file
DEBUG: Parsed JSON successfully, type: <class 'dict'>
DEBUG: Loaded 14 places from JSON
DEBUG: Sample place_ids: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
DEBUG: Saving 14 places to JSON file: backend/data/places_bootstrap.json
DEBUG: Sample place_ids being saved: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
DEBUG: First place structure keys: ['place_id', 'places_details_flag', 'enriched_flag', 'place', 'sources', 'derived', 'places_details_called_at', 'places_details_called_version', 'enriched_at', 'enriched_version']
DEBUG: First place flags - places_details: True, enriched: False
DEBUG: Successfully saved 396231 bytes to file
Reset 2 places


In [9]:
test_cfg = Config(
    api_key=PLACES_API_KEY,
    lat=40.76894303040112,
    lng=-73.98811816889815,
    radius_m=500,  # Smaller radius for testing
    grid_step_m=400,
    nearby_search_radius_m=300,
    include_types=("cafe",),
    keyword=None,
    max_places_to_enrich=2,  # Small number for testing
    request_sleep_s=0.25,
)


In [3]:
# Test 1: Sync operations - nearby_search with auto-enrichment
# This will:
# 1. Run nearby_search
# 2. Upsert place_ids to JSON
# 3. Optionally fetch place_details if places_details_flag is false


# Run nearby_search with sync enrichment
results = nearby_search_with_sync(test_cfg, place_type="cafe", auto_enrich_sync=True)

print(f"Found {len(results)} places")
print(f"First place: {results[0].get('name') if results else 'None'}")


NameError: name 'nearby_search_with_sync' is not defined

In [5]:
# Check what was saved to JSON
places = load_places_json(JSON_PATH)
print(f"Total places in JSON: {len(places)}")

# Show first place
if places:
    first_place_id = list(places.keys())[0]
    first_place = places[first_place_id]
    print(f"\nFirst place ID: {first_place_id}")
    print(f"Places details flag: {first_place.get('places_details_flag')}")
    print(f"Enriched flag: {first_place.get('enriched_flag')}")
    print(f"Place name: {first_place.get('place', {}).get('name', 'N/A')}")
    print(f"Has reviews: {len(first_place.get('sources', {}).get('google_reviews', {}).get('reviews', []))} reviews")


DEBUG: File exists, size: 381617 bytes
DEBUG: Read 378979 characters from file
DEBUG: Parsed JSON successfully, type: <class 'dict'>
DEBUG: Loaded 14 places from JSON
DEBUG: Sample place_ids: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
Total places in JSON: 14

First place ID: ChIJ1bM7WeVZwokRdnontikwFK4
Places details flag: True
Enriched flag: False
Place name: The Oasis Cafe
Has reviews: 5 reviews


In [23]:
# Test 2: Async operations - Tavily + LLM enrichment
# This will:
# 1. Run Tavily search for places where enriched_flag is false
# 2. Use LLM to derive attributes from Google reviews + Tavily results
# 3. Save derived attributes to JSON

# Set JSON path
JSON_PATH = "backend/data/places_bootstrap.json"

# Get places needing enrichment
needing_enrichment = get_places_needing_enrichment(JSON_PATH)
print(f"Places needing async enrichment: {len(needing_enrichment)}")

# Run async enrichment for first few places (limit for testing)
if needing_enrichment:
    test_place_ids = needing_enrichment[:2]  # Just test with 2 places
    print(f"Running async enrichment for {len(test_place_ids)} places...")
    
    enriched_count = process_enrichment_async(test_cfg, place_ids=test_place_ids, json_path=JSON_PATH)
    print(f"Enriched {enriched_count} places")
else:
    print("No places need enrichment")


DEBUG: File exists, size: 396231 bytes
DEBUG: Read 393583 characters from file
DEBUG: Parsed JSON successfully, type: <class 'dict'>
DEBUG: Loaded 14 places from JSON
DEBUG: Sample place_ids: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
Places needing async enrichment: 14
Running async enrichment for 2 places...
DEBUG: File exists, size: 396231 bytes
DEBUG: Read 393583 characters from file
DEBUG: Parsed JSON successfully, type: <class 'dict'>
DEBUG: Loaded 14 places from JSON
DEBUG: Sample place_ids: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']


LLM returned has_outlets='unknown' with confidence 0.0 but no evidence. Overriding to 'unknown' with confidence 0.0
VALIDATION FAILED: seating_comfort='mixed' (conf=0.5) has evidence that doesn't match attribute keywords. Evidence: ["Review: 'the environment blends clean decor with a relaxing vibe'"]. Keywords required: ['comfortable', 'comfort', 'uncomfortable', 'cushion', 'hard', 'soft', 'ergonomic', 'chair', 'seating']. Overriding to 'unknown' with confidence 0.0


DEBUG: [upsert_place] Upserting place: ChIJ1bM7WeVZwokRdnontikwFK4
DEBUG: File exists, size: 396231 bytes
DEBUG: Read 393583 characters from file
DEBUG: Parsed JSON successfully, type: <class 'dict'>
DEBUG: Loaded 14 places from JSON
DEBUG: Sample place_ids: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
DEBUG: [upsert_place] Place already exists
DEBUG: Saving 14 places to JSON file: backend/data/places_bootstrap.json
DEBUG: Sample place_ids being saved: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
DEBUG: First place structure keys: ['place_id', 'places_details_flag', 'enriched_flag', 'place', 'sources', 'derived', 'places_details_called_at', 'places_details_called_version', 'enriched_at', 'enriched_version']
DEBUG: First place flags - places_details: True, enriched: True
DEBUG: Successfully saved 397242 bytes to file
DEBUG: [upsert_place] Successfully upserted: ChIJ1bM7WeVZwokRdnontikwFK4


Failed to generate queries with LLM: Expecting value: line 1 column 1 (char 0), using fallback
LLM returned has_wifi='unknown' with confidence 0.0 but no evidence. Overriding to 'unknown' with confidence 0.0
LLM returned has_outlets='unknown' with confidence 0.0 but no evidence. Overriding to 'unknown' with confidence 0.0
VALIDATION FAILED: noise_level='mixed' (conf=0.5) has evidence that doesn't match attribute keywords. Evidence: ["Review 1: 'too many dogs inside this small coffee shop'", "Review 4: 'The atmosphere of this location is second to none.'"]. Keywords required: ['noise', 'noisy', 'quiet', 'quietly', 'loud', 'peaceful', 'calm', 'busy', 'crowded', 'music', 'sound']. Overriding to 'unknown' with confidence 0.0
LLM returned seating_comfort='unknown' with confidence 0.0 but no evidence. Overriding to 'unknown' with confidence 0.0


DEBUG: [upsert_place] Upserting place: ChIJW6K4fFlYwokRl9znZwmIvEU
DEBUG: File exists, size: 397242 bytes
DEBUG: Read 394594 characters from file
DEBUG: Parsed JSON successfully, type: <class 'dict'>
DEBUG: Loaded 14 places from JSON
DEBUG: Sample place_ids: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
DEBUG: [upsert_place] Place already exists
DEBUG: Saving 14 places to JSON file: backend/data/places_bootstrap.json
DEBUG: Sample place_ids being saved: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
DEBUG: First place structure keys: ['place_id', 'places_details_flag', 'enriched_flag', 'place', 'sources', 'derived', 'places_details_called_at', 'places_details_called_version', 'enriched_at', 'enriched_version']
DEBUG: First place flags - places_details: True, enriched: True
DEBUG: Successfully saved 397298 bytes to file
DEBUG: [upsert_place] Successfully upserted: ChIJW6K4fFlYwokRl9znZwmIvEU
E

In [24]:
# Verify async enrichment results
places = load_places_json(JSON_PATH)

# Check if any places have been enriched
enriched_places = [pid for pid, place in places.items() if place.get('enriched_flag')]

if enriched_places:
    test_place_id = enriched_places[0]
    test_place = places[test_place_id]
    
    print(f"Enriched place: {test_place.get('place', {}).get('name')}")
    print(f"\nTavily sources:")
    tavily = test_place.get('sources', {}).get('tavily', {})
    print(f"  Query: {tavily.get('query')}")
    print(f"  Results: {len(tavily.get('results', []))} results")
    print(f"  Excerpts: {len(tavily.get('excerpts', []))} excerpts")
    
    print(f"\nDerived attributes:")
    derived = test_place.get('derived', {})
    for attr_name, attr_data in derived.items():
        if isinstance(attr_data, dict):
            value = attr_data.get('value', 'N/A')
            confidence = attr_data.get('confidence', 0.0)
            sources_count = len(attr_data.get('sources', []))
            evidence_count = len(attr_data.get('evidence', []))
            print(f"  {attr_name}: {value} (confidence: {confidence:.2f}, sources: {sources_count}, evidence: {evidence_count})")
else:
    print("No places have been enriched yet")


DEBUG: File exists, size: 397298 bytes
DEBUG: Read 394650 characters from file
DEBUG: Parsed JSON successfully, type: <class 'dict'>
DEBUG: Loaded 14 places from JSON
DEBUG: Sample place_ids: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
Enriched place: The Oasis Cafe

Tavily sources:
  Query: The Oasis Cafe 857 9th Ave remote work wifi power outlets reviews
  Results: 8 results
  Excerpts: 8 excerpts

Derived attributes:
  has_wifi: free (confidence: 0.70, sources: 1, evidence: 1)
  has_outlets: unknown (confidence: 0.00, sources: 0, evidence: 0)
  is_laptop_friendly: mixed (confidence: 0.50, sources: 1, evidence: 1)
  noise_level: mixed (confidence: 0.50, sources: 1, evidence: 1)
  seating_availability: limited (confidence: 0.70, sources: 1, evidence: 1)
  seating_comfort: unknown (confidence: 0.00, sources: 0, evidence: 0)
  notable_positives: ['Delicious desserts', 'Unique pastries', 'Friendly staff', 'Cozy atmosphere', 'Good coffee']

In [25]:
# Test 3: Verify all content is saved to sources
# Check that Google reviews, Tavily results, and excerpts are all saved

places = load_places_json(JSON_PATH)
if places:
    test_place_id = list(places.keys())[1]
    test_place = places[test_place_id]
    sources = test_place.get('sources', {})
    
    print(f"Place: {test_place.get('place', {}).get('name')}")
    print(f"\nSources saved:")
    print(f"  google_details: {'Yes' if 'google_details' in sources else 'No'}")
    print(f"  google_reviews: {'Yes' if 'google_reviews' in sources else 'No'}")
    if 'google_reviews' in sources:
        reviews = sources['google_reviews'].get('reviews', [])
        print(f"    - {len(reviews)} reviews saved")
    print(f"  tavily: {'Yes' if 'tavily' in sources else 'No'}")
    if 'tavily' in sources:
        tavily = sources['tavily']
        print(f"    - Query: {tavily.get('query', 'N/A')}")
        print(f"    - Results: {len(tavily.get('results', []))} results")
        print(f"    - Excerpts: {len(tavily.get('excerpts', []))} excerpts")


DEBUG: File exists, size: 397298 bytes
DEBUG: Read 394650 characters from file
DEBUG: Parsed JSON successfully, type: <class 'dict'>
DEBUG: Loaded 14 places from JSON
DEBUG: Sample place_ids: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
Place: Rex

Sources saved:
  google_details: Yes
  google_reviews: Yes
    - 5 reviews saved
  tavily: Yes
    - Query: Is Rex in 864 10th Ave, New York, NY 10019, USA good for working laptop wifi outlets
    - Results: 8 results
    - Excerpts: 8 excerpts
